# TIỂU LUẬN CUỐI KHÓA - ĐỀ TÀI 7
## MÔ HÌNH CHẤM ĐIỂM TÍN DỤNG & THẨM ĐỊNH RỦI RO VỠ NỢ (CREDIT SCORING & LOAN DEFAULT PREDICTION)
---
**Môn học:** Dữ Liệu Lớn & Trí Tuệ Nhân Tạo Trong Kinh Doanh  
**Môi trường:** Databricks Community / Serverless Edition (PySpark, Spark SQL, Spark MLlib, Delta Lake)  
**Kiến trúc:** Medallion Architecture (Bronze -> Silver -> Gold)

In [ ]:
# =========================================================
# CELL 1: BRONZE LAYER - INGESTION & DELTA STORAGE
# =========================================================
from pyspark.sql import SparkSession

# 1. Đọc trực tiếp từ Bảng Managed Table mà Databricks đã lưu khi upload GUI
df_raw = spark.table("workspace.default.credit_risk_dataset")

print("--- CẤU TRÚC DỮ LIỆU THÔ (RAW SCHEMA) ---")
df_raw.printSchema()

# Hiển thị 5 dòng đầu tiên
display(df_raw.limit(5))

# 2. Lưu dữ liệu thô vào Delta Table chuẩn Bronze Layer
df_raw.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_credit_data")

print("✅ ĐÃ NẠP THÀNH CÔNG DỮ LIỆU VÀO BRONZE DELTA TABLE: bronze_credit_data")

In [ ]:
# =========================================================
# VISUALIZATION CELL 1: BRONZE LAYER (DATA QUALITY & MISSING VALUES)
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, count, when

df_bronze = spark.table("workspace.default.bronze_credit_data")
total_rows = df_bronze.count()

missing_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns]
missing_df = df_bronze.select(missing_exprs).toPandas().T
missing_df.columns = ["Missing_Count"]
missing_df["Missing_Percentage"] = (missing_df["Missing_Count"] / total_rows) * 100
missing_df = missing_df.sort_values(by="Missing_Percentage", ascending=False)

plt.figure(figsize=(10, 4))
ax = sns.barplot(x=missing_df.index, y=missing_df["Missing_Percentage"], palette="Reds_r")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Tỷ lệ thiếu (%)")
plt.title("BRONZE LAYER: TỶ LỆ DỮ LIỆU KHUYẾT THIẾU (MISSING VALUES)", fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
display(plt.show())

In [ ]:
# =========================================================
# CELL 2: SILVER LAYER - DISTRIBUTED CLEANING & PREPROCESSING
# =========================================================
from pyspark.sql.functions import col, when

# 1. Đọc dữ liệu từ Bronze Table
df_bronze = spark.table("workspace.default.bronze_credit_data")

# 2. Làm sạch dữ liệu & Xử lý giá trị bị thiếu (Imputation)
df_silver = df_bronze \
    .filter((col("person_age") >= 18) & (col("person_age") <= 80)) \
    .filter(col("person_income") > 0) \
    .na.fill({
        "person_emp_length": 0,
        "loan_int_rate": 11.0,
        "cb_person_cred_hist_length": 1
    })

# 3. Chuẩn hóa kiểu dữ liệu
df_silver = df_silver \
    .withColumn("person_income", col("person_income").cast("double")) \
    .withColumn("loan_amnt", col("loan_amnt").cast("double")) \
    .withColumn("loan_status", col("loan_status").cast("integer"))

# 4. Lưu vào Silver Delta Table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_credit_data")

print("✅ ĐÃ HOÀN THÀNH LÀM SẠCH VÀ LƯU VÀO SILVER DELTA TABLE: silver_credit_data")
display(df_silver.describe())

In [ ]:
# =========================================================
# VISUALIZATION CELL 2: SILVER LAYER (NPL RATE ANALYSIS)
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns

df_silver_pdf = spark.table("workspace.default.silver_credit_data") \
    .groupBy("loan_intent", "person_home_ownership") \
    .agg({"loan_status": "avg"}) \
    .toPandas()

df_silver_pdf["NPL_Rate_Pct"] = df_silver_pdf["avg(loan_status)"] * 100

plt.figure(figsize=(11, 5))
sns.barplot(data=df_silver_pdf, x="loan_intent", y="NPL_Rate_Pct", hue="person_home_ownership", palette="Set2")
plt.title("SILVER LAYER: TỶ LỆ NỢ XẤU (NPL RATE %) THEO MỤC ĐÍCH VAY VÀ SỞ HỮU NHÀ", fontweight="bold")
plt.xlabel("Mục đích vay")
plt.ylabel("Tỷ lệ Nợ xấu (%)")
plt.xticks(rotation=30)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
display(plt.show())

In [ ]:
# =========================================================
# CELL 3: SPARK SQL ANALYTICS & KPIS COMPUTATION
# =========================================================
df_sql_kpi = spark.sql("""
SELECT 
    loan_intent AS Muc_Dich_Vay,
    COUNT(*) AS Tong_So_Khoan_Vay,
    SUM(loan_status) AS So_Khoan_Vay_Vo_No,
    ROUND(AVG(loan_status) * 100, 2) AS Ty_Le_Vo_No_Pct,
    ROUND(AVG(loan_amnt), 0) AS Khoan_Vay_Trung_Binh,
    ROUND(AVG(loan_int_rate), 2) AS Lai_Suat_Trung_Binh
FROM workspace.default.silver_credit_data
GROUP BY loan_intent
ORDER BY Ty_Le_Vo_No_Pct DESC
""")

display(df_sql_kpi)

In [ ]:
# =========================================================
# CELL 3.1: SPARK SQL WINDOW FUNCTIONS (DTI & RANKING)
# =========================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, round, col

df_silver = spark.table("workspace.default.silver_credit_data")
windowSpec = Window.partitionBy("loan_intent").orderBy(col("person_income").desc())

df_kpi = df_silver \
    .withColumn("income_rank", rank().over(windowSpec)) \
    .withColumn("debt_to_income_ratio", round(col("loan_amnt") / col("person_income"), 4))

display(df_kpi.select("person_age", "person_income", "loan_intent", "loan_amnt", "debt_to_income_ratio", "income_rank").limit(10))

In [ ]:
# =========================================================
# CELL 4: GOLD LAYER - SPARK MLLIB PIPELINE & CROSS VALIDATION
# =========================================================
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

data = spark.table("workspace.default.silver_credit_data")
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

cat_cols = ["person_home_ownership", "loan_intent", "cb_person_default_on_file"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in cat_cols]
encoders = [OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_vec") for c in cat_cols]

num_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt", "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length"]
feature_cols = [f"{c}_vec" for c in cat_cols] + num_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

rf = RandomForestClassifier(labelCol="loan_status", featuresCol="features", seed=42)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, rf])

paramGrid = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [5, 10]) \
    .addGrid(rf.numTrees, [20, 50]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="loan_status", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print("⏳ Đang huấn luyện mô hình Spark MLlib với CrossValidation...")
cv_model = cv.fit(train_data)

predictions = cv_model.transform(test_data)
auc = evaluator.evaluate(predictions)
print("==================================================")
print(f"📊 CHỈ SỐ ROC-AUC ĐẠT ĐƯỢC CỦA MÔ HÌNH: {auc:.4f}")
print("==================================================")

predictions.select(
    "person_age", "person_income", "loan_amnt", "loan_intent", 
    "loan_status", "prediction", "probability"
).write.format("delta") \
 .mode("overwrite") \
 .saveAsTable("workspace.default.gold_credit_scoring")

print("✅ ĐÃ LƯU BẢNG KẾT QUẢ DỰ BÁO VÀO GOLD DELTA TABLE: gold_credit_scoring")

In [ ]:
# =========================================================
# VISUALIZATION CELL 3: GOLD LAYER (RISK SEGMENTATION & DECISIONS)
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, when, udf
from pyspark.sql.types import DoubleType

extract_prob = udf(lambda v: float(v[1]), DoubleType())
df_gold = spark.table("workspace.default.gold_credit_scoring").withColumn("default_probability", extract_prob(col("probability")))

df_decision = df_gold.withColumn("Risk_Segment", 
    when(col("default_probability") < 0.15, "1. Rủi ro Thấp")
    .when(col("default_probability") < 0.35, "2. Rủi ro Trung Bình")
    .when(col("default_probability") < 0.60, "3. Rủi ro Cao")
    .otherwise("4. Rủi ro Rất Cao")
).withColumn("Credit_Policy",
    when(col("Risk_Segment") == "1. Rủi ro Thấp", "Duyệt Tự Động")
    .when(col("Risk_Segment") == "2. Rủi ro Trung Bình", "Duyệt Hạn Mức 70%")
    .when(col("Risk_Segment") == "3. Rủi ro Cao", "Thẩm Định Thủ Công")
    .otherwise("Từ Chối Vay (Reject)")
)

df_summary = df_decision.groupBy("Risk_Segment", "Credit_Policy").count().toPandas().sort_values(by="Risk_Segment")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

ax1.pie(df_summary['count'], labels=df_summary['Risk_Segment'], autopct='%1.1f%%', startangle=140, colors=colors)
ax1.add_artist(plt.Circle((0, 0), 0.50, fc='white'))
ax1.set_title("GOLD LAYER: CƠ CẤU PHÂN TẦNG RỦI RO TÍN DỤNG", fontweight="bold")

sns.barplot(data=df_summary, y="Credit_Policy", x="count", palette=colors, ax=ax2)
ax2.set_title("QUYẾT ĐỊNH PHÊ DUYỆT TÍN DỤNG TỰ ĐỘNG", fontweight="bold")
plt.tight_layout()
display(plt.show())

In [ ]:
# =========================================================
# CELL 5: BUSINESS DECISION ENGINE & STRATEGIC RECOMMENDATIONS
# =========================================================
display(df_decision.select("person_income", "loan_amnt", "default_probability", "Risk_Segment", "Credit_Policy").limit(10))

In [ ]:
# =========================================================
# VISUALIZATION CELL 4: CONFUSION MATRIX & MODEL EVALUATION REPORT
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

predictions_pd = spark.table("workspace.default.gold_credit_scoring").select("loan_status", "prediction").toPandas()
cm = confusion_matrix(predictions_pd["loan_status"], predictions_pd["prediction"])

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Trả đúng hạn (0)", "Vỡ nợ (1)"],
            yticklabels=["Trả đúng hạn (0)", "Vỡ nợ (1)"],
            annot_kws={"size": 13, "weight": "bold"})

plt.title("MA TRẬN NHẦM LẪN (CONFUSION MATRIX)", fontweight="bold")
plt.xlabel("Dự Báo Mô Hình")
plt.ylabel("Thực Tế")
plt.tight_layout()
display(plt.show())

print("--- BÁO CÁO CHI TIẾT HIỆU NĂNG MÔ HÌNH ---")
print(classification_report(predictions_pd["loan_status"], predictions_pd["prediction"]))